In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from scipy.spatial.transform import Rotation as R

plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "cm"

# Helper Functions

In [ ]:
u = np.arange(-np.pi / 1.5, np.pi / 1.5, 0.15)
v = np.arange(-np.pi / 1.5, np.pi / 1.5, 0.15)
U, V = np.meshgrid(u, v)

h = .2



Let $ q_1 = \frac{u}{n} $, $ q_2 = \frac{v}{n} $, where $ n $ is the number of joints. The link length is:

$
\ell = \frac{h}{2n}
$

Each joint transformation matrix $ \mathbf{G}(q_1, q_2) \in SE(3) $ for a pair of rotations (yaw $ q_1 $, then pitch $ q_2 $) followed by a translation of length $ \ell $ along the final rotated frame is defined as:

$
\mathbf{G}(q_1, q_2) =
\begin{bmatrix}
\cos q_2 & 0 & \sin q_2 & \ell \sin q_2 \\
\sin q_1 \sin q_2 & \cos q_1 & -\sin q_1 \cos q_2 & -\ell \sin q_1 \cos q_2 \\
-\cos q_1 \sin q_2 & \sin q_1 & \cos q_1 \cos q_2 & \ell \cos q_1 \cos q_2 + \ell \\
0 & 0 & 0 & 1
\end{bmatrix}
$

The total transformation after $ n $ such segments is:

$
\mathbf{T}_{\text{total}} = \left(\mathbf{G}(q_1, q_2)\right)^n
$

see universal_joint_fk_derivation.pdf


In [ ]:
def calc_ee_loc(u, v):
    phi = np.sqrt(u**2 + v**2)
    if phi.any() < 1e-6:
        return np.array([v * h / 2, -u * h / 2, h - h * phi**2 / 6])
    else:
        rho_x = -h * v / phi**2
        rho_y = -h * u / phi**2
        rho = np.sqrt(rho_x**2 + rho_y**2)
        sigma = np.cos(phi) - 1
        x = rho_x * sigma
        y = -rho_y * sigma
        z = rho * np.sin(phi)
        return np.array([x, y, z])


def calc_ee_orientation(u, v):
    if np.isscalar(u):
        u = np.array([u]).reshape(-1, 1)
    if np.isscalar(v):
        v = np.array([v]).reshape(-1, 1)
        
    r = np.zeros((3, 3, u.shape[0], u.shape[1]))
    phi = np.sqrt(u**2 + v**2)
    sigma = np.cos(phi) - 1
    if phi.any() < 1e-6:
        r[0, 0] = 1 - v**2 / 2
        r[0, 1] = u * v / 2
        r[0, 2] = v
        r[1, 0] = u * v / 2
        r[1, 1] = 1 - u**2 / 2
        r[1, 2] = -u
        r[2, 0] = -v
        r[2, 1] = u
        r[2, 2] = 1 - phi**2 / 2
    else:
        vtilde = v / phi
        utilde = u / phi
        r[0, 0] = sigma * vtilde**2 + 1
        r[0, 1] = -sigma * vtilde * utilde
        r[0, 2] = vtilde * np.sin(phi)
        r[1, 0] = -sigma * vtilde * utilde
        r[1, 1] = sigma * utilde**2 + 1
        r[1, 2] = -utilde * np.sin(phi)
        r[2, 0] = -vtilde * np.sin(phi)
        r[2, 1] = utilde * np.sin(phi)
        r[2, 2] = np.cos(phi)

    return r



In [ ]:
def calc_universal_FK(u, v, num_joints=1):
    # see universal_joint_fk_derivation.pdf

    # u,v are lists of angles for each set of x and y universal joints
    l = h / (num_joints * 2)

    # make blank homogeneous transformation matrix
    result = np.eye(4)
    for i in range(num_joints):

        q1 = u[i]
        q2 = v[i]

        g = np.zeros((4, 4))

        g[0, 0] = np.cos(q2)
        g[0, 2] = np.sin(q2)
        g[0, 3] = l * np.sin(q2)

        g[1, 0] = np.sin(q1) * np.sin(q2)
        g[1, 1] = np.cos(q1)
        g[1, 2] = -np.sin(q1) * np.cos(q2)
        g[1, 3] = -l * np.sin(q1) * np.cos(q2)

        g[2, 0] = -np.cos(q1) * np.sin(q2)
        g[2, 1] = np.sin(q1)
        g[2, 2] = np.cos(q1) * np.cos(q2)
        g[2, 3] = l * np.cos(q1) * np.cos(q2) + l

        g[3, 3] = 1


        result = np.matmul(result, g)

    return result


def calc_error(loc1, loc2):
    return np.linalg.norm(loc1 - loc2, axis=0)


def R2vec(r):
    r = R.from_matrix(r)
    return R.as_rotvec(r)


def calcOrientationError(r1, r2):
    # 3x3xUxV
    error = np.zeros((r1.shape[2], r1.shape[3]))
    for each in range(r1.shape[2]):
        for each2 in range(r1.shape[3]):
            vec1 = R2vec(r1[:, :, each, each2])
            vec2 = R2vec(r2[:, :, each, each2])
            error[each, each2] = np.linalg.norm(vec1 - vec2)

    return error


# Inverse Kinematics

By running IK to solve for the set of universal joint angles to minimzie the positino and orientation error between it and the CC end effector pose, the remaining residual error is due only to the limitations in the kinematic model of the universal joint. 

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation as R

def error_function(angles, cc_angles, num_joints=1, lambda_orientation=1.0):
    # Unpack joint angles
    u = cc_angles[0]
    v = cc_angles[1]

    u_guesses = angles[:num_joints]
    v_guesses = angles[num_joints:]
    
    # Compute FK with current guess
    g_current = calc_universal_FK(u_guesses, v_guesses, num_joints)
    p_current = g_current[0:3, 3].reshape(3, 1)
    R_current = g_current[0:3, 0:3]

    # Compute target pose
    p_target = calc_ee_loc(u, v).reshape(3, 1)
    R_target = calc_ee_orientation(u, v).reshape(3, 3)

    # Position error
    e_p = p_current - p_target

    # Orientation error (rotation vector from R_current^T * R_target)
    R_err = R_current.T @ R_target
    rotvec = R.from_matrix(R_err).as_rotvec().reshape(3, 1)  # shape (3, 1)

    # Total 6D pose error vector
    pose_error = np.vstack((e_p, lambda_orientation * rotvec))  # shape (6, 1)

    # Compute total error norm (squared for smoothness)
    E_total = np.linalg.norm(pose_error)**2

    # Add small regularization on joint angle magnitudes
    reg = .05 * np.linalg.norm(angles)**2

    return E_total + reg

from scipy.optimize import minimize

def solve_ik(angles, num_joints=1, initial_guess=None, lambda_orientation=1.0):
    # If initial guess is None, start with a guess of zeros
    if initial_guess is None:
        initial_guess = np.zeros(num_joints * 2)
    
    # Call the optimizer to minimize the error function
    result = minimize(
        error_function, 
        initial_guess, 
        args=(angles, num_joints, lambda_orientation), 
        method='BFGS',
        options={'disp': False, 'maxiter': 1000}
        )
    
    # print("Optimization result:", result)
    
    # Extract the optimized joint angles (u, v) from the result
    if result.success:
        optimized_angles = result.x
        u_opt = optimized_angles[:num_joints]
        v_opt = optimized_angles[num_joints:]
        return u_opt, v_opt
    else:
        raise ValueError("Optimization failed: " + result.message)

# Example usage
angles = [-2,2]

# Call the IK solver
num_joints = 63
initial_guess_x = [angles[0] / 63] * num_joints
initial_guess_y = [angles[1] / 63] * num_joints
initial_guess = np.concatenate((initial_guess_x, initial_guess_y))

u_opt, v_opt = solve_ik(angles, num_joints=num_joints, initial_guess=initial_guess, lambda_orientation=.5)

print("Optimized universal joint angles:")
print("u:\n", np.round(u_opt, 3))
print("v:\n", np.round(v_opt, 3))


constant_curvature_ee_pos = calc_ee_loc(angles[0], angles[1]).flatten()
constant_curvature_ee_ori = calc_ee_orientation(angles[0], angles[1]).reshape(3,3)

universal_g = calc_universal_FK(u_opt, v_opt, num_joints=num_joints).reshape(4, 4)
universal_ee_pos = universal_g[0:3, 3]
universal_ee_ori = universal_g[0:3, 0:3]

print("Constant curvature end-effector position:", constant_curvature_ee_pos)
print("Universal end-effector position:", universal_ee_pos)

print("Difference in end-effector position:", np.linalg.norm(constant_curvature_ee_pos - universal_ee_pos))
print("Difference in end-effector orientation (angle):", np.arccos((np.trace(np.dot(constant_curvature_ee_ori.T, universal_ee_ori)) - 1) / 2))


## Generate Data on the grid

In [ ]:
#this one takes a while

import numpy as np
import matplotlib.pyplot as plt
plt.close('all')

u = np.arange(-np.pi / 1.5, np.pi / 1.5, 0.25)
v = np.arange(-np.pi / 1.5, np.pi / 1.5, 0.25)
U, V = np.meshgrid(u, v, indexing='ij')

num_joints_list = [1, 3, 7, 15, 31, 63]
all_pos_errors = []
all_ori_errors = []

from tqdm.notebook import tqdm  # Add tqdm for progress bars

# First pass to compute all errors and find global max
for num_joints in tqdm(num_joints_list, desc="Number of Joints", leave=True, position=0):
    ee_pos_errors = np.zeros_like(U)
    ee_ori_errors = np.zeros_like(U)

    for i, ui in enumerate(tqdm(u, desc="U Loop", leave=False, position=1)):
        for j, vj in enumerate(tqdm(v, desc="V Loop", leave=False, position=2)):
            constant_curvature_ee_pos = calc_ee_loc(ui, vj).flatten()
            constant_curvature_ee_ori = calc_ee_orientation(ui, vj).reshape(3, 3)

            initial_guess_x = [ui / num_joints] * num_joints
            initial_guess_y = [vj / num_joints] * num_joints
            initial_guess = np.concatenate((initial_guess_x, initial_guess_y))

            u_opt, v_opt = solve_ik(
                [ui, vj],
                num_joints=num_joints,
                initial_guess=initial_guess,
                lambda_orientation=.5,
            )

            universal_g = calc_universal_FK(u_opt, v_opt, num_joints=num_joints).reshape(4, 4)
            universal_ee_pos = universal_g[0:3, 3]
            universal_ee_ori = universal_g[0:3, 0:3]

            pos_error = np.linalg.norm(constant_curvature_ee_pos - universal_ee_pos)
            ori_error = np.arccos(np.clip((np.trace(constant_curvature_ee_ori.T @ universal_ee_ori) - 1) / 2, -1.0, 1.0))

            ee_pos_errors[i, j] = pos_error
            ee_ori_errors[i, j] = ori_error

    all_pos_errors.append(ee_pos_errors)
    all_ori_errors.append(ee_ori_errors)

# Plotting

In [ ]:

# Fixed vmin = 0, find global vmax
pos_vmax = max(np.max(e) for e in all_pos_errors)
ori_vmax = max(np.max(e) for e in all_ori_errors)

# Create plots
fig, axs = plt.subplots(len(num_joints_list), 2, figsize=(10, 4 * len(num_joints_list)))

for row, num_joints in enumerate(num_joints_list):
    cp1 = axs[row, 0].contourf(U, V, all_pos_errors[row], cmap='viridis', vmin=0, vmax=pos_vmax)
    axs[row, 0].set_title(f'Position Error (m) $N$={num_joints+1}')
    axs[row, 0].set_xlabel('$q_0$')
    axs[row, 0].set_ylabel('$q_1$')
    fig.colorbar(cp1, ax=axs[row, 0])

    cp2 = axs[row, 1].contourf(U, V, all_ori_errors[row], cmap='plasma', vmin=0, vmax=ori_vmax)
    axs[row, 1].set_title(f'Orientation Error (rad) $N$={num_joints+1}')
    axs[row, 1].set_xlabel('$q_0$')
    axs[row, 1].set_ylabel('$q_1$')
    fig.colorbar(cp2, ax=axs[row, 1])

plt.tight_layout()
plt.show()



In [ ]:
# Use Set1 colormap for distinct bright colors
colors = plt.cm.Set1.colors

# Position Error Plot
fig_pos = plt.figure(figsize=(8, 8))
ax1 = fig_pos.add_subplot(1, 1, 1, projection='3d')
for i, errors in enumerate(all_pos_errors):
    c = ax1.plot_surface(U, V, errors, alpha=0.7, edgecolor='white', linewidth=0.1, shade=False, label=f'$N$={num_joints_list[i]+1}')
    c._facecolors2d = c._facecolor3d
    c._edgecolors2d = c._edgecolor3d

ax1.set_xlabel('$q_0$', fontsize=16, labelpad=15)
ax1.set_ylabel('$q_1$', fontsize=16, labelpad=15)
ax1.set_zlabel('Position Error (m)', fontsize=16, labelpad=20)
ax1.set_xticks([-2, -1, 0, 1, 2])
ax1.set_yticks([-2, -1, 0, 1, 2])
ax1.tick_params(axis='both', which='major', labelsize=14, pad=10)
ax1.legend(fontsize=16, loc='upper left')
# fig_pos.tight_layout()
ax1.set_box_aspect(None, zoom=0.8)
fig_pos.savefig("position_error_surface.png", bbox_inches='tight', pad_inches=0.1, dpi=300)

# Orientation Error Plot
fig_ori = plt.figure(figsize=(8, 8))
ax2 = fig_ori.add_subplot(1, 1, 1, projection='3d')
for i, errors in enumerate(all_ori_errors):
    c = ax2.plot_surface(U, V, errors, alpha=0.7, edgecolor='white', linewidth=0.1, shade=False, label=f'$N$={num_joints_list[i]+1}')
    c._facecolors2d = c._facecolor3d
    c._edgecolors2d = c._edgecolor3d

ax2.set_xlabel('$q_0$', fontsize=16, labelpad=15)
ax2.set_ylabel('$q_1$', fontsize=16, labelpad=15)
ax2.set_zlabel('Orientation Error (rad)', fontsize=16, labelpad=20)
ax2.set_xticks([-2, -1, 0, 1, 2])
ax2.set_yticks([-2, -1, 0, 1, 2])
ax2.tick_params(axis='both', which='major', labelsize=14, pad=10)
ax2.legend(fontsize=16, loc='upper left')
ax2.set_box_aspect(None, zoom=0.8)
fig_ori.savefig("orientation_error_surface.png", bbox_inches='tight', pad_inches=0.5, dpi=300)

plt.show()


In [ ]:
# Create box plots for position errors
plt.figure(figsize=(8, 8))
box_pos = plt.boxplot(pos_errors_for_boxplot, labels=labels, showmeans=True)
plt.xlabel("Number of Disks $N$", fontsize=16)
plt.ylabel("Position Error (m)", fontsize=16)
plt.grid()
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.ylim(0, 0.13)
for i, mean in enumerate([np.mean(errors) for errors in pos_errors_for_boxplot]):
    max_val = np.max(pos_errors_for_boxplot[i])
    plt.text(i + 1, max_val + 0.002, f'{mean:.4f}', ha='center', va='bottom', fontsize=16)
plt.tight_layout()
plt.savefig("position_error_boxplot.png", bbox_inches='tight', dpi=300)

# Create box plots for orientation errors
plt.figure(figsize=(8, 8))
box_ori = plt.boxplot(ori_errors_for_boxplot, labels=labels, showmeans=True)
plt.xlabel("Number of Disks $N$", fontsize=16)
plt.ylabel("Orientation Error (rad)", fontsize=16)
plt.grid()
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.ylim(0, 1.6)
for i, mean in enumerate([np.mean(errors) for errors in ori_errors_for_boxplot]):
    max_val = np.max(ori_errors_for_boxplot[i])
    plt.text(i + 1, max_val + 0.02, f'{mean:.4f}', ha='center', va='bottom', fontsize=16)
plt.tight_layout()
plt.savefig("orientation_error_boxplot.png", bbox_inches='tight', dpi=300)

plt.show()

# Old

In [ ]:
ee_loc = calc_ee_loc(U, V)
ee_orientation = calc_ee_orientation(U, V)  # 3x3xUxV

uni_pose = calc_universal_FK(U, V, 1)
uni3_pose = calc_universal_FK(U, V, 3)
uni7_pose = calc_universal_FK(U, V, 7)
uni15_pose = calc_universal_FK(U, V, 15)
uni31_pose = calc_universal_FK(U, V, 31)
uni63_pose = calc_universal_FK(U, V, 63)

uni_loc = uni_pose[:3, 3]
uni3_loc = uni3_pose[:3, 3]
uni7_loc = uni7_pose[:3, 3]
uni15_loc = uni15_pose[:3, 3]
uni31_loc = uni31_pose[:3, 3]
uni63_loc = uni63_pose[:3, 3]

uni_orientation = uni_pose[:3, :3]
uni3_orientation = uni3_pose[:3, :3]
uni7_orientation = uni7_pose[:3, :3]
uni15_orientation = uni15_pose[:3, :3]
uni31_orientation = uni31_pose[:3, :3]
uni63_orientation = uni63_pose[:3, :3]

error1 = calc_error(ee_loc, uni_loc)
error3 = calc_error(ee_loc, uni3_loc)
error7 = calc_error(ee_loc, uni7_loc)
error15 = calc_error(ee_loc, uni15_loc)
error31 = calc_error(ee_loc, uni31_loc)
error63 = calc_error(ee_loc, uni63_loc)

error_orientation1 = calcOrientationError(ee_orientation, uni_orientation)
error_orientation3 = calcOrientationError(ee_orientation, uni3_orientation)
error_orientation7 = calcOrientationError(ee_orientation, uni7_orientation)
error_orientation15 = calcOrientationError(ee_orientation, uni15_orientation)
error_orientation31 = calcOrientationError(ee_orientation, uni31_orientation)
error_orientation63 = calcOrientationError(ee_orientation, uni63_orientation)

In [ ]:
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(U, V, error1, label='$N$=2')
ax.scatter(U, V, error3, label='$N$=4')
ax.scatter(U, V, error7, label='$N$=8')
ax.scatter(U, V, error15, label='$N$=16')
ax.scatter(U, V, error31, label='$N$=32')
ax.scatter(U, V, error63, label='$N$=64')
ax.set_xlabel('U')
ax.set_ylabel('V')
ax.set_zlabel('Error (m)')
ax.legend()
plt.savefig('tip_error_surface.png', dpi=300)
plt.show()


In [ ]:
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(U, V, error_orientation1, label='$N$=2')
ax.scatter(U, V, error_orientation3, label='$N$=4')
ax.scatter(U, V, error_orientation7, label='$N$=8')
ax.scatter(U, V, error_orientation15, label='$N$=16')
ax.scatter(U, V, error_orientation31, label='$N$=32')
ax.scatter(U, V, error_orientation63, label='$N$=64')
ax.set_xlabel('U')
ax.set_ylabel('V')
ax.set_zlabel('Error (rad)')
ax.legend()
plt.savefig('orientation_error_surface.png', dpi=300)
plt.show()


In [ ]:
#make box plot of error stats for each number of universal joints
plt.figure(figsize=(6,6))
plt.boxplot([
    error1.flatten(),
    error3.flatten(),
    error7.flatten(),
    error15.flatten(),
    error31.flatten(),
    error63.flatten()
])
#label box plot x axis with number of universal joints
plt.xticks([1, 2, 3, 4, 5, 6], ['2', '4', '8', '16', '32', '64'])
plt.xlabel('Number of disks ($N$)')
plt.ylabel('Tip Error (m)')
plt.grid("true")

#save to 300 dpi
plt.savefig('tip_error_box.png', dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(6, 6))
plt.boxplot([
    error_orientation1.flatten(),
    error_orientation3.flatten(),
    error_orientation7.flatten(),
    error_orientation15.flatten(),
    error_orientation31.flatten(),
    error_orientation63.flatten(),
])
#label box plot x axis with number of universal joints
plt.xticks([1, 2, 3, 4, 5, 6], ['2', '4', '8', '16', '32', '64'])
plt.xlabel('Number of disks ($N$)')
plt.ylabel('Tip Orientation Error (rad)')
plt.grid("true")
plt.savefig('orientation_error_box.png', dpi=300)
plt.show()



In [ ]:
#print min, max, median for each box
print('Tip Error Stats')
print('2: min: {}, max: {}, median: {}'.format(np.min(error1), np.max(error1),
                                             np.median(error1)))
print('4: min: {}, max: {}, median: {}'.format(np.min(error3), np.max(error3),
                                             np.median(error3)))
print('8: min: {}, max: {}, median: {}'.format(np.min(error7), np.max(error7),
                                             np.median(error7)))
print('16: min: {}, max: {}, median: {}'.format(np.min(error15), np.max(error15),
                                              np.median(error15)))
print('32: min: {}, max: {}, median: {}'.format(np.min(error31), np.max(error31),
                                              np.median(error31)))
print('64: min: {}, max: {}, median: {}'.format(np.min(error63), np.max(error63),
                                              np.median(error63)))

print('Tip Orientation Error Stats')
print('2: min: {}, max: {}, median: {}'.format(np.min(error_orientation1), np.max(error_orientation1),
                                             np.median(error_orientation1)))
print('4: min: {}, max: {}, median: {}'.format(np.min(error_orientation3), np.max(error_orientation3),
                                             np.median(error_orientation3)))
print('8: min: {}, max: {}, median: {}'.format(np.min(error_orientation7), np.max(error_orientation7),
                                             np.median(error_orientation7)))
print('16: min: {}, max: {}, median: {}'.format(np.min(error_orientation15), np.max(error_orientation15),
                                              np.median(error_orientation15)))
print('32: min: {}, max: {}, median: {}'.format(np.min(error_orientation31), np.max(error_orientation31),
                                              np.median(error_orientation31)))
print('64: min: {}, max: {}, median: {}'.format(np.min(error_orientation63), np.max(error_orientation63),
                                              np.median(error_orientation63)))

In [ ]:
# Plot universal joint location vs continuum location
%matplotlib widget
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot continuum location
ax.plot_surface(ee_loc[0], ee_loc[1], ee_loc[2], color='r', alpha=0.6)

# Plot universal joint locations for different numbers of joints
ax.scatter(uni_loc[0], uni_loc[1], uni_loc[2],  label='Universal Joint (N=2)', alpha=0.6)
ax.scatter(uni3_loc[0], uni3_loc[1], uni3_loc[2], label='Universal Joint (N=4)', alpha=0.6)
ax.scatter(uni7_loc[0], uni7_loc[1], uni7_loc[2],  label='Universal Joint (N=8)', alpha=0.6)
ax.scatter(uni15_loc[0], uni15_loc[1], uni15_loc[2],  label='Universal Joint (N=16)', alpha=0.6)
ax.scatter(uni31_loc[0], uni31_loc[1], uni31_loc[2],  label='Universal Joint (N=32)', alpha=0.6)
ax.scatter(uni63_loc[0], uni63_loc[1], uni63_loc[2],  label='Universal Joint (N=64)', alpha=0.6)

# Set labels and legend
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.legend()

# Save and show the plot
plt.savefig('universal_vs_continuum_3d_plot.png', dpi=300)
plt.show()